# BERT baseline on Colab (needs a GPU)
Runtime ▸ Change runtime type ▸ **T4 GPU**. Each student is written as a sentence and a BERT model is fine-tuned on the leakage-free hold-out split, then compared with the ensemble (McNemar + bootstrap).

In [1]:
REPO_URL = "https://github.com/ntvy243/dropout-prediction"
import os
if not os.path.exists('/content/dropout-prediction'):
    !git clone $REPO_URL /content/dropout-prediction
%cd /content/dropout-prediction
!pip install -q -r requirements-bert.txt
!python src/download_data.py

Cloning into '/content/dropout-prediction'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 175 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 8.34 MiB | 14.78 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/dropout-prediction
data/data.csv already present


In [2]:
!nvidia-smi | head -5

Thu Nov 19 03:00:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [3]:
!python src/run_bert.py --seeds 3

example: marital status is 1. application mode is 17. application order is 5. course is 171. daytime/evening attendance is daytime. previous qualification is 1. previous qualification grade is 122. nacionality is 1. mother's qualification is 19. father's qualification is 12. mother's occupation is 5. father' ...
fit 3185 val 354 hold-out 885 device cuda
config.json: 100% 570/570 [00:00<00:00, 2.74MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 249kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 11.0MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 43.7MB/s]
token length: mean 235, max 240, truncated at 384: 0.0%

model.safetensors: downloading bytes:  88% 388M/440M [00:01<00:00, 308MB/s, 33.3MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 139MB/s, 37.1MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 148MB/s, 40.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5381.89it/s]
[transformers] BertForSequenceClassification LOA

In [4]:
import pandas as pd
display(pd.read_csv('results/bert_holdout.csv').round(4))
display(pd.read_csv('results/bert_vs_ensemble.csv').round(4))

,model,accuracy,macro_f1,f1_Dropout,f1_Enrolled,f1_Graduate,recall_Dropout,recall_Enrolled,recall_Graduate,precision_Dropout,precision_Enrolled,precision_Graduate
0,bert-base-uncased (seed 0),0.7266,0.6856,0.7290,0.4925,0.8351,0.6584,0.6226,0.8077,0.8166,0.4074,0.8644
1,bert-base-uncased (seed 1),0.7141,0.6708,0.7006,0.4816,0.8303,0.6056,0.6163,0.8190,0.8309,0.3952,0.8419
2,bert-base-uncased (seed 2),0.7006,0.6684,0.7239,0.4784,0.8028,0.6232,0.6604,0.7647,0.8634,0.3750,0.8450
3,bert-base-uncased MEAN over 3 seeds,0.7138,0.6749,0.7179,0.4842,0.8227,0.6291,0.6331,0.7971,0.8370,0.3925,0.8504
4,SD over seeds,0.0130,0.0093,0.0152,0.0074,0.0174,0.0269,0.0238,0.0286,0.0240,0.0164,0.0122
5,Proposed ensemble (same split),0.7548,0.7130,0.7601,0.5337,0.8451,0.6972,0.6226,0.8394,0.8354,0.4670,0.8509


,metric,ensemble_minus_bert,ci_low,ci_high,mcnemar_exact_p_accuracy
0,accuracy,0.0282,0.0034,0.0542,0.0314
1,macro_f1,0.0272,0.0012,0.0559,0.0314
2,f1_Enrolled,0.0411,-0.0045,0.0914,0.0314
3,recall_Enrolled,-0.0006,-0.0650,0.0690,0.0314
